# OffScript — Phase 3: Deviation Analysis

Measuring how individual pitchers deviate from the baseline pitch 
selection theory and determining whether those deviations are 
effective or costly.

## Imports

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import joblib
from sklearn.metrics import accuracy_score
from pitch_analysis import load_clean_data, get_pitcher

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

print("Imports successful")

## Load Model and Data

In [ ]:
# Load clean dataset
data = load_clean_data()
print(f"Dataset loaded: {len(data)} pitches")

# Load saved models
model = joblib.load('../models/baseline_pitch_model.pkl')
le = joblib.load('../models/label_encoder.pkl')
pitcher_encoder = joblib.load('../models/pitcher_encoder.pkl')

print(f"Model loaded successfully")
print(f"Pitch classes: {le.classes_}")
print(f"Pitchers encoded: {pitcher_encoder.classes_}")

## Reconstruct Feature Matrix

In [ ]:
# Reconstruct all engineered features — must match Phase 2 exactly
data['stand_encoded'] = (data['stand'] == 'R').astype(int)
data['runners_on'] = (data['on_1b'].fillna(0) + 
                      data['on_2b'].fillna(0) + 
                      data['on_3b'].fillna(0))
data['scoring_position'] = (
    (data['on_2b'].fillna(0) + data['on_3b'].fillna(0)) > 0
).astype(int)
data['count_leverage'] = (
    (data['strikes'] == 2).astype(int) * 2 +
    (data['balls'] == 3).astype(int) * 2 +
    (data['strikes'] == 1).astype(int) +
    (data['balls'] == 2).astype(int)
)
data['pitcher_encoded'] = pitcher_encoder.transform(data['pitcher_name'])

feature_cols = [
    'balls', 'strikes', 'inning', 'score_diff',
    'on_1b', 'on_2b', 'on_3b', 'runners_on',
    'scoring_position', 'stand_encoded',
    'pitcher_encoded', 'count_leverage'
]

print("Features reconstructed successfully")
print(f"Feature columns: {feature_cols}")

## Generate Model Recommendations

In [ ]:
# Run every pitch through the model to get recommendations
X_all = data[feature_cols].fillna(0)
data['recommended_pitch'] = le.inverse_transform(model.predict(X_all))

# Get probability for each pitch type
proba = model.predict_proba(X_all)
proba_df = pd.DataFrame(proba, columns=le.classes_, index=data.index)

# Add recommended pitch probability — confidence of recommendation
data['recommendation_confidence'] = proba.max(axis=1)

# Flag whether pitcher followed the recommendation
data['followed_recommendation'] = (
    data['pitch_type'] == data['recommended_pitch']
)

print("=== Recommendation Generation Complete ===")
print(f"Total pitches analyzed: {len(data)}")
print(f"\nOverall recommendation follow rate:")
print(f"{data['followed_recommendation'].mean():.3f}")
print(f"\nPer pitcher follow rate:")
print(data.groupby('pitcher_name')['followed_recommendation']
      .mean()
      .sort_values()
      .round(3))

## Calculate Formal Deviation Scores

In [ ]:
# Calculate per pitcher deviation metrics
deviation_summary = data.groupby('pitcher_name').agg(
    total_pitches=('pitch_type', 'count'),
    follow_rate=('followed_recommendation', 'mean'),
    avg_confidence=('recommendation_confidence', 'mean')
).reset_index()

# Deviation score — inverse of follow rate, scaled 0-100
deviation_summary['deviation_score'] = (
    (1 - deviation_summary['follow_rate']) * 100
).round(1)

# Rank pitchers by deviation
deviation_summary = deviation_summary.sort_values(
    'deviation_score', ascending=False
).reset_index(drop=True)
deviation_summary['deviation_rank'] = range(1, len(deviation_summary) + 1)

print("=== Pitcher Deviation Scores ===")
print(deviation_summary[['pitcher_name', 'total_pitches', 
                          'deviation_score', 'avg_confidence',
                          'deviation_rank']].to_string(index=False))

## Visualize Deviation Scores

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#d62728' if s > 55 else '#1f77b4' if s < 45 else '#ff7f0e' 
          for s in deviation_summary['deviation_score']]

bars = ax.barh(deviation_summary['pitcher_name'], 
               deviation_summary['deviation_score'],
               color=colors)

# Add value labels on bars
for bar, val in zip(bars, deviation_summary['deviation_score']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=10)

# Add league average line
avg_deviation = deviation_summary['deviation_score'].mean()
ax.axvline(x=avg_deviation, color='black', linestyle='--', 
           linewidth=1.5, label=f'League Avg: {avg_deviation:.1f}')

ax.set_xlabel("Deviation Score (higher = deviates more from model)", 
              fontsize=11)
ax.set_title("OffScript — Pitcher Deviation Scores\n"
             "Red = High Deviation | Blue = Low Deviation | "
             "Orange = Near Average", fontsize=13)
ax.legend()
ax.set_xlim(0, 80)
plt.tight_layout()
plt.savefig('../reports/figures/deviation_scores.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Deviation by Count State

In [ ]:
# Deviation rate by count state for each pitcher
count_deviation = data.groupby(
    ['pitcher_name', 'count']
)['followed_recommendation'].mean().reset_index()
count_deviation['deviation_rate'] = (
    1 - count_deviation['followed_recommendation']
)

# Focus on your top 5 deviators for clarity
top_deviators = deviation_summary.head(5)['pitcher_name'].tolist()
count_dev_top = count_deviation[
    count_deviation['pitcher_name'].isin(top_deviators)
]

# Pivot for heatmap
count_pivot = count_dev_top.pivot(
    index='pitcher_name', 
    columns='count', 
    values='deviation_rate'
).fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(count_pivot,
            annot=True,
            fmt='.2f',
            cmap='RdYlGn_r',
            ax=ax,
            vmin=0,
            vmax=1)
ax.set_title("Deviation Rate by Count State — Top 5 Deviators\n"
             "(1.0 = never follows recommendation, "
             "0.0 = always follows)", fontsize=12)
ax.set_xlabel("Count State")
ax.set_ylabel("Pitcher")
plt.tight_layout()
plt.savefig('../reports/figures/deviation_by_count.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Deviation by Batter Handedness

In [ ]:
# Does deviation pattern change based on batter handedness?
hand_deviation = data.groupby(
    ['pitcher_name', 'stand']
)['followed_recommendation'].mean().reset_index()
hand_deviation['deviation_rate'] = (
    1 - hand_deviation['followed_recommendation']
)

# Filter to top deviators
hand_dev_top = hand_deviation[
    hand_deviation['pitcher_name'].isin(top_deviators)
]

fig, ax = plt.subplots(figsize=(10, 6))
hand_pivot = hand_dev_top.pivot(
    index='pitcher_name',
    columns='stand',
    values='deviation_rate'
)
hand_pivot.plot(kind='bar', ax=ax, color=['#1f77b4', '#d62728'],
                width=0.6)
ax.set_title("Deviation Rate by Batter Handedness — Top 5 Deviators",
             fontsize=12)
ax.set_ylabel("Deviation Rate")
ax.set_xlabel("Pitcher")
ax.legend(title="Batter Stands", labels=['Left', 'Right'])
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/deviation_by_handedness.png',
            dpi=150, bbox_inches='tight')
plt.show()

## What Pitches They Throw Instead

In [ ]:
# For deviating pitches only — what did they throw vs what was recommended
deviations_only = data[~data['followed_recommendation']].copy()

def deviation_substitution(pitcher_name):
    """Show what a pitcher throws instead of the recommendation."""
    pdf = deviations_only[deviations_only['pitcher_name'] == pitcher_name]
    
    substitution = pdf.groupby(
        ['recommended_pitch', 'pitch_type']
    ).size().reset_index(name='count')
    
    substitution['pct'] = (
        substitution.groupby('recommended_pitch')['count']
        .transform(lambda x: x / x.sum() * 100)
        .round(1)
    )
    
    return substitution.sort_values(
        ['recommended_pitch', 'count'], ascending=[True, False]
    )

# Print substitution patterns for top deviators
for name in top_deviators:
    print(f"\n=== {name} — Deviation Substitutions ===")
    print(f"(When model recommends X, pitcher throws Y)")
    sub = deviation_substitution(name)
    print(sub[sub['pct'] > 10].to_string(index=False))

## Hypothesis Generation

## Initial Deviation Hypotheses

### Gerrit Cole
Cole systematically avoids his knuckle curve (KC) in situations where 
the model recommends it, substituting his fastball (55.6%) and slider 
(22.5%) instead. Simultaneously he replaces recommended fastballs with 
sliders 43.2% of the time. This suggests Cole treats his slider as his 
primary putaway pitch rather than his knuckle curve, potentially 
reflecting greater confidence in his slider's consistency at the elite 
level or a deliberate deception strategy of using his fastball to set 
up the slider rather than the knuckle curve.

### Marcus Stroman
The model frequently recommends a splitter (FS) that Stroman largely 
avoids, substituting fastball (51.2%), cutter (23.1%), and curveball 
(20.6%) instead. Stroman is known primarily as a sinker/ground ball 
pitcher whose arsenal is built around fastball variants and a curveball 
rather than a splitter. This deviation likely reflects arsenal 
composition — the model may be recommending a pitch Stroman throws 
infrequently or with limited effectiveness.

### Nestor Cortes
Cortes shows a strong preference for his fastball over the cutter and 
sweeper that the model frequently recommends. Given his unorthodox 
delivery and deceptive mechanics, this may reflect a philosophy of 
letting his release point do the work rather than relying on pitch 
movement variation. His fastball may play up significantly beyond 
its raw metrics due to his unusual arm angle.

### Zack Wheeler
Wheeler substitutes his four seam fastball for his sinker in 73.5% 
of sinker-recommended situations. This likely reflects Wheeler using 
his fastball and sinker as situational variants of the same pitch 
concept — working down in the zone with either offering depending on 
feel and batter tendencies rather than strict situational logic.

### Yusei Kikuchi
Kikuchi shows a contradictory pattern — avoiding fastballs in fastball 
situations while over-relying on fastballs in breaking ball situations. 
This may reflect a significant mid-career philosophical shift in his 
approach, as Kikuchi dramatically reinvented himself during this period 
moving toward a slider-heavy attack. The model may be capturing his 
transitional tendencies rather than a settled approach.

## Deviation Effectiveness Analysis

In [ ]:
# Define outcome effectiveness
# Positive outcomes for pitcher: strikeout, swinging strike, called strike
# Negative outcomes: hit, walk, home run

positive_outcomes = [
    'strikeout', 'swinging_strike', 'called_strike',
    'swinging_strike_blocked', 'foul', 'foul_tip'
]

negative_outcomes = [
    'single', 'double', 'triple', 'home_run', 
    'walk', 'hit_by_pitch'
]

def classify_outcome(row):
    if row['description'] in ['swinging_strike', 'called_strike',
                               'swinging_strike_blocked']:
        return 'positive'
    elif row['description'] in ['ball', 'blocked_ball']:
        return 'negative'
    elif row['events'] in ['strikeout', 'strikeout_double_play']:
        return 'positive'
    elif row['events'] in ['single', 'double', 'triple', 
                            'home_run', 'walk', 'hit_by_pitch']:
        return 'negative'
    else:
        return 'neutral'

data['outcome_class'] = data.apply(classify_outcome, axis=1)

# Compare effectiveness when following vs deviating
effectiveness = data.groupby(
    ['pitcher_name', 'followed_recommendation', 'outcome_class']
).size().reset_index(name='count')

effectiveness['pct'] = (
    effectiveness.groupby(['pitcher_name', 'followed_recommendation'])
    ['count'].transform(lambda x: x / x.sum() * 100)
)

# Pivot to compare positive outcome rates
positive_rate = effectiveness[
    effectiveness['outcome_class'] == 'positive'
].pivot_table(
    index='pitcher_name',
    columns='followed_recommendation',
    values='pct'
).round(2)

positive_rate.columns = ['deviation_positive_rate', 'followed_positive_rate']
positive_rate['deviation_advantage'] = (
    positive_rate['deviation_positive_rate'] - 
    positive_rate['followed_positive_rate']
).round(2)

positive_rate = positive_rate.sort_values('deviation_advantage', 
                                           ascending=False)

print("=== Deviation Effectiveness ===")
print("Positive outcome rate when deviating vs following recommendation")
print("Positive deviation_advantage = deviating works better")
print("Negative deviation_advantage = following recommendation works better")
print()
print(positive_rate.to_string())

## Visualize Effectiveness

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#2ca02c' if v > 0 else '#d62728' 
          for v in positive_rate['deviation_advantage']]

bars = ax.barh(positive_rate.index, 
               positive_rate['deviation_advantage'],
               color=colors)

for bar, val in zip(bars, positive_rate['deviation_advantage']):
    offset = 0.3 if val >= 0 else -0.3
    ax.text(val + offset, 
            bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%', 
            va='center', fontsize=10)

ax.axvline(x=0, color='black', linewidth=1.5)
ax.set_xlabel("Deviation Advantage (% positive outcomes: deviation vs recommendation)",
              fontsize=11)
ax.set_title("OffScript — Is Deviating From the Model Effective?\n"
             "Green = Deviation helps | Red = Following model is better",
             fontsize=13)
plt.tight_layout()
plt.savefig('../reports/figures/deviation_effectiveness.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Per Pitcher Detailed Effectiveness

In [ ]:
# Deep dive on your top deviators
for name in top_deviators:
    pitcher_data = data[data['pitcher_name'] == name]
    
    dev_positive = pitcher_data[
        (~pitcher_data['followed_recommendation']) & 
        (pitcher_data['outcome_class'] == 'positive')
    ].shape[0]
    dev_total = pitcher_data[
        ~pitcher_data['followed_recommendation']
    ].shape[0]
    
    follow_positive = pitcher_data[
        (pitcher_data['followed_recommendation']) & 
        (pitcher_data['outcome_class'] == 'positive')
    ].shape[0]
    follow_total = pitcher_data[
        pitcher_data['followed_recommendation']
    ].shape[0]
    
    dev_rate = dev_positive / dev_total * 100 if dev_total > 0 else 0
    follow_rate = follow_positive / follow_total * 100 if follow_total > 0 else 0
    
    print(f"\n=== {name} ===")
    print(f"When deviating:   {dev_rate:.1f}% positive outcomes "
          f"({dev_positive}/{dev_total} pitches)")
    print(f"When following:   {follow_rate:.1f}% positive outcomes "
          f"({follow_positive}/{follow_total} pitches)")
    print(f"Deviation advantage: {dev_rate - follow_rate:+.1f}%")

## Effectiveness Findings

## Deviation Effectiveness Findings

### Overarching Finding
For 13 of 14 pitchers analyzed, following the model recommendation 
produces better outcomes than deviating from it, validating the 
baseline pitch selection theory. However the magnitude of the 
deviation penalty varies dramatically across pitchers, suggesting 
that some deviations are more strategic than others.

### Three Tiers of Deviation Impact

**Tier 1 — Costless Deviators (0 to -1% advantage)**
Burnes, Strider, Hendricks, Valdez, Wheeler deviate without 
meaningful cost. Their elite stuff or sophisticated sequencing 
likely compensates for unconventional selection.

**Tier 2 — Moderately Costly Deviators (-1% to -3%)**
Cortes, Verlander, Scherzer, Cease, Webb leave measurable value 
on the table but remain effective overall.

**Tier 3 — Significantly Costly Deviators (-3% and beyond)**
Stroman, Cole, Kikuchi, and Sale show the largest gaps between 
deviation and recommendation effectiveness. Sale's -10.74% gap 
is the most dramatic finding and warrants deeper investigation.

### Notable Exceptions
- **Corbin Burnes (+0.79%)** is the only pitcher where deviating 
  outperforms the model, suggesting genuine strategic sophistication
  beyond what the baseline theory captures.
- **Chris Sale (-10.74%)** represents an extreme outlier whose 
  deviation patterns appear significantly costly.

## Chris Sale Deep Dive

In [ ]:
# Deep dive on Chris Sale — most dramatic deviation penalty
sale_data = data[data['pitcher_name'] == 'Chris Sale'].copy()

print("=== Chris Sale — Complete Profile ===")
print(f"Total pitches: {len(sale_data)}")
print(f"\nPitch Arsenal:")
print(sale_data['pitch_type'].value_counts())
print(f"\nOverall follow rate: {sale_data['followed_recommendation'].mean():.3f}")

# Where specifically is Sale deviating and getting hurt?
sale_dev = sale_data[~sale_data['followed_recommendation']].copy()
sale_follow = sale_data[sale_data['followed_recommendation']].copy()

# Outcome breakdown for deviating pitches
print("\n=== Sale Deviation Outcomes ===")
print(sale_dev['outcome_class'].value_counts(normalize=True).round(3) * 100)

print("\n=== Sale Following Recommendation Outcomes ===")
print(sale_follow['outcome_class'].value_counts(normalize=True).round(3) * 100)

# Which specific deviations are most costly?
sale_dev_outcomes = sale_data.groupby(
    ['recommended_pitch', 'pitch_type', 'outcome_class']
).size().reset_index(name='count')

sale_dev_outcomes['pct'] = (
    sale_dev_outcomes.groupby(['recommended_pitch', 'pitch_type'])
    ['count'].transform(lambda x: x / x.sum() * 100)
).round(1)

# Focus on negative outcomes from deviations
sale_costly = sale_dev_outcomes[
    sale_dev_outcomes['outcome_class'] == 'negative'
].sort_values('count', ascending=False)

print("\n=== Sale Most Costly Deviation Patterns ===")
print("(Recommended → Actual → Negative outcome frequency)")
print(sale_costly.head(15).to_string(index=False))

## Corbin Burnes Deep Dive

In [ ]:
# Deep dive on Corbin Burnes — only pitcher where deviation helps
burnes_data = data[data['pitcher_name'] == 'Corbin Burnes'].copy()

print("=== Corbin Burnes — Deviation Success Analysis ===")
print(f"Total pitches: {len(burnes_data)}")
print(f"Follow rate: {burnes_data['followed_recommendation'].mean():.3f}")
print(f"\nPitch Arsenal:")
print(burnes_data['pitch_type'].value_counts())

# What situations does Burnes deviate in successfully?
burnes_dev = burnes_data[~burnes_data['followed_recommendation']].copy()

# Deviation success by count
burnes_count_dev = burnes_dev.groupby('count').agg(
    total=('outcome_class', 'count'),
    positive=('outcome_class', lambda x: (x == 'positive').sum())
).reset_index()
burnes_count_dev['positive_rate'] = (
    burnes_count_dev['positive'] / 
    burnes_count_dev['total'] * 100
).round(1)

print("\n=== Burnes Deviation Success by Count ===")
print(burnes_count_dev.sort_values(
    'positive_rate', ascending=False
).to_string(index=False))

# What is Burnes substituting successfully?
print("\n=== Burnes Successful Deviation Substitutions ===")
burnes_sub = burnes_dev[burnes_dev['outcome_class'] == 'positive'].groupby(
    ['recommended_pitch', 'pitch_type']
).size().reset_index(name='successful_deviations')
print(burnes_sub.sort_values(
    'successful_deviations', ascending=False
).head(10).to_string(index=False))

## Situational Deviation Analysis

In [ ]:
# For all pitchers — when in the game are deviations most costly?
situational = data.groupby(
    ['pitcher_name', 'followed_recommendation', 'balls', 'strikes']
).apply(lambda x: pd.Series({
    'positive_rate': (x['outcome_class'] == 'positive').mean() * 100,
    'pitch_count': len(x)
})).reset_index()

# Focus on two strike counts — highest leverage situations
two_strike = situational[situational['strikes'] == 2].copy()

two_strike_pivot = two_strike.pivot_table(
    index='pitcher_name',
    columns='followed_recommendation',
    values='positive_rate'
).round(2)

two_strike_pivot.columns = ['deviation_rate', 'follow_rate']
two_strike_pivot['two_strike_deviation_cost'] = (
    two_strike_pivot['deviation_rate'] - 
    two_strike_pivot['follow_rate']
).round(2)

print("=== Two Strike Count Deviation Analysis ===")
print("(Most critical situation — does deviation hurt more here?)")
print(two_strike_pivot.sort_values(
    'two_strike_deviation_cost'
).to_string())

## Save Deviation Results

In [ ]:
import os

# Save deviation summary for use in future notebooks and API
os.makedirs('../data/processed', exist_ok=True)

# Save main deviation summary
deviation_summary.to_parquet(
    '../data/processed/deviation_summary.parquet', 
    index=False
)

# Save full data with recommendations attached
data.to_parquet(
    '../data/processed/pitcher_data_with_recommendations.parquet',
    index=False
)

# Save positive_rate comparison
positive_rate.to_parquet(
    '../data/processed/deviation_effectiveness.parquet'
)

print("=== Deviation Analysis Results Saved ===")
print("deviation_summary.parquet")
print("pitcher_data_with_recommendations.parquet")
print("deviation_effectiveness.parquet")

## Early vs Late Count Deviation Breakdown

In [ ]:
# Split deviation analysis into early counts vs two strike counts
# This reveals whether deviations are strategic sequencing or mistakes

def count_category(row):
    if row['strikes'] == 2:
        return 'two_strike'
    elif row['balls'] >= 3:
        return 'hitter_count'
    elif row['strikes'] == 0 and row['balls'] == 0:
        return 'first_pitch'
    else:
        return 'middle_count'

data['count_category'] = data.apply(count_category, axis=1)

# Calculate deviation cost by count category for each pitcher
count_cat_analysis = data.groupby(
    ['pitcher_name', 'count_category', 'followed_recommendation']
).apply(lambda x: pd.Series({
    'positive_rate': (x['outcome_class'] == 'positive').mean() * 100,
    'pitch_count': len(x)
})).reset_index()

# Pivot to get deviation advantage by count category
cat_pivot = count_cat_analysis.pivot_table(
    index=['pitcher_name', 'count_category'],
    columns='followed_recommendation',
    values='positive_rate'
).reset_index()

cat_pivot.columns = ['pitcher_name', 'count_category', 
                     'deviation_rate', 'follow_rate']
cat_pivot['deviation_cost'] = (
    cat_pivot['deviation_rate'] - cat_pivot['follow_rate']
).round(2)

# Focus on your most interesting pitchers
interesting_pitchers = [
    'Gerrit Cole', 'Chris Sale', 'Corbin Burnes', 
    'Dylan Cease', 'Yusei Kikuchi'
]

print("=== Deviation Cost by Count Category ===")
print("Negative = following model is better")
print("Positive = deviating is better\n")

for name in interesting_pitchers:
    pitcher_cat = cat_pivot[
        cat_pivot['pitcher_name'] == name
    ][['count_category', 'deviation_cost', 
       'deviation_rate', 'follow_rate']].set_index('count_category')
    print(f"\n{name}:")
    print(pitcher_cat.round(2).to_string())

## Dylan Cease Two Strike Investigation

In [ ]:
# Cease two strike deviation is surprisingly costly given moderate overall deviation
cease_data = data[data['pitcher_name'] == 'Dylan Cease'].copy()
cease_two_strike = cease_data[cease_data['strikes'] == 2].copy()

print("=== Dylan Cease — Two Strike Analysis ===")
print(f"Total two strike pitches: {len(cease_two_strike)}")
print(f"\nTwo strike pitch arsenal:")
print(cease_two_strike['pitch_type'].value_counts())

print(f"\nTwo strike follow rate: "
      f"{cease_two_strike['followed_recommendation'].mean():.3f}")

# What does Cease throw vs recommend in two strike counts
cease_ts_sub = cease_two_strike.groupby(
    ['recommended_pitch', 'pitch_type']
).size().reset_index(name='count')

cease_ts_sub['pct'] = (
    cease_ts_sub.groupby('recommended_pitch')['count']
    .transform(lambda x: x / x.sum() * 100)
).round(1)

print("\n=== Two Strike Substitutions ===")
print(cease_ts_sub[cease_ts_sub['pct'] > 10]
      .sort_values(['recommended_pitch', 'count'], ascending=[True, False])
      .to_string(index=False))

# Outcome by pitch type in two strike counts
print("\n=== Two Strike Outcomes by Pitch Type ===")
cease_ts_outcomes = cease_two_strike.groupby(
    ['pitch_type', 'outcome_class']
).size().reset_index(name='count')
cease_ts_outcomes['pct'] = (
    cease_ts_outcomes.groupby('pitch_type')['count']
    .transform(lambda x: x / x.sum() * 100)
).round(1)
print(cease_ts_outcomes[cease_ts_outcomes['outcome_class'] == 'positive']
      .sort_values('pct', ascending=False)
      .to_string(index=False))

## Final Pitcher Profile Summary

In [ ]:
# Combine all deviation metrics into one comprehensive pitcher profile
pitcher_profiles = deviation_summary.merge(
    positive_rate.reset_index(), 
    on='pitcher_name'
).merge(
    two_strike_pivot.reset_index().rename(columns={
        'two_strike_deviation_cost': 'two_strike_dev_cost'
    })[['pitcher_name', 'two_strike_dev_cost']],
    on='pitcher_name'
)

# Add arsenal size
arsenal_size = data.groupby('pitcher_name')['pitch_type'].nunique().reset_index()
arsenal_size.columns = ['pitcher_name', 'arsenal_size']
pitcher_profiles = pitcher_profiles.merge(arsenal_size, on='pitcher_name')

pitcher_profiles = pitcher_profiles[[
    'pitcher_name', 'total_pitches', 'deviation_score',
    'deviation_positive_rate', 'followed_positive_rate',
    'deviation_advantage', 'two_strike_dev_cost', 'arsenal_size'
]].sort_values('deviation_advantage')

print("=== Complete Pitcher Deviation Profiles ===")
print(pitcher_profiles.round(2).to_string(index=False))

# Save comprehensive profiles
pitcher_profiles.to_parquet(
    '../data/processed/pitcher_profiles.parquet',
    index=False
)
print("\nProfiles saved to data/processed/pitcher_profiles.parquet")

## Phase 3 Complete — Deviation Analysis Summary

### Core Finding
The baseline pitch selection theory is validated — following model 
recommendations produces better outcomes for 13 of 14 pitchers analyzed.
However deviation cost varies dramatically, revealing three distinct 
pitcher archetypes.

### Pitcher Archetypes Identified

**The Strategic Deviator (Burnes, Wheeler, Scherzer)**
These pitchers deviate profitably or at near-zero cost, particularly 
in two strike counts. Their deviations appear to reflect genuine 
tactical sophistication — recognizing when batters are anticipating 
the obvious pitch and successfully countering expectations.

**The Costly Deviator (Sale, Kikuchi, Stroman, Cole)**
These pitchers pay a significant price for deviating from optimal 
selection. Sale's -10.74% overall and -10.46% two strike deviation 
cost is the most dramatic finding — his slider avoidance when the 
model recommends it is consistently punished.

**The Neutral Deviator (Strider, Hendricks, Cortes, Wheeler)**
These pitchers deviate without meaningful cost in most situations, 
suggesting their raw stuff or command compensates for suboptimal 
pitch selection.

### Key Individual Findings
- **Cole:** Costly overall deviator but skilled two strike pitcher,
  suggesting early count deviations are deliberate sequencing
- **Sale:** Consistent deviation penalty across all count categories,
  slider avoidance is primary cost driver  
- **Burnes:** Only pitcher where deviation helps, driven by 
  successfully substituting breaking balls for over-recommended cutters
- **Cease:** Moderate overall deviator but significant two strike 
  deviation cost — a hidden vulnerability in high leverage situations

### Phase 4 Preview
These pitcher profiles now feed directly into batter vulnerability 
mapping — if Sale avoids his slider in certain situations, batters 
who struggle against fastballs become significantly more dangerous 
matchups against him in those scenarios.

# Phase 3 Visualization Capstone

## The Quadrant Chart

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

# Plot each pitcher as a point
for _, row in pitcher_profiles.iterrows():
    ax.scatter(row['deviation_score'], row['deviation_advantage'],
              s=row['total_pitches'] / 40,
              alpha=0.7,
              zorder=5)
    ax.annotate(row['pitcher_name'],
               (row['deviation_score'], row['deviation_advantage']),
               textcoords="offset points",
               xytext=(8, 4),
               fontsize=9)

# Quadrant lines
avg_dev_score = pitcher_profiles['deviation_score'].mean()
ax.axhline(y=0, color='black', linewidth=1.5, linestyle='--')
ax.axvline(x=avg_dev_score, color='black', linewidth=1.5, linestyle='--')

# Quadrant labels
ax.text(35, 1.2, 'LOW DEVIATION\nPOSITIVE OUTCOME',
        fontsize=9, color='green', alpha=0.7, style='italic')
ax.text(57, 1.2, 'HIGH DEVIATION\nPOSITIVE OUTCOME',
        fontsize=9, color='green', alpha=0.7, style='italic')
ax.text(35, -10.5, 'LOW DEVIATION\nNEGATIVE OUTCOME',
        fontsize=9, color='red', alpha=0.7, style='italic')
ax.text(57, -10.5, 'HIGH DEVIATION\nNEGATIVE OUTCOME',
        fontsize=9, color='red', alpha=0.7, style='italic')

# Shading
ax.axhspan(0, ax.get_ylim()[1] if ax.get_ylim()[1] > 2 else 2,
           alpha=0.05, color='green')
ax.axhspan(ax.get_ylim()[0] if ax.get_ylim()[0] < -1 else -12,
           0, alpha=0.05, color='red')

ax.set_xlabel("Deviation Score (higher = deviates more from model)",
              fontsize=11)
ax.set_ylabel("Deviation Advantage % (positive = deviating helps)",
              fontsize=11)
ax.set_title("OffScript — Pitcher Deviation Quadrant Analysis\n"
             "Bubble size = total pitches analyzed",
             fontsize=13)

plt.tight_layout()
plt.savefig('../reports/figures/deviation_quadrant.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Two Strike vs Overall Deviation Cost Comparison

In [ ]:
# Compare overall deviation cost vs two strike deviation cost
# Reveals pitchers whose two strike approach differs from overall pattern

comparison_data = pitcher_profiles[[
    'pitcher_name', 'deviation_advantage', 'two_strike_dev_cost'
]].copy()

fig, ax = plt.subplots(figsize=(12, 7))

x = range(len(comparison_data))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x],
               comparison_data['deviation_advantage'],
               width, label='Overall Deviation Advantage',
               color='steelblue', alpha=0.8)

bars2 = ax.bar([i + width/2 for i in x],
               comparison_data['two_strike_dev_cost'],
               width, label='Two Strike Deviation Cost',
               color='darkorange', alpha=0.8)

ax.axhline(y=0, color='black', linewidth=1.5)
ax.set_xticks(list(x))
ax.set_xticklabels(comparison_data['pitcher_name'],
                   rotation=35, ha='right', fontsize=9)
ax.set_ylabel("Deviation Advantage % (negative = model is better)")
ax.set_title("OffScript — Overall vs Two Strike Deviation Cost\n"
             "Orange bars above blue = pitcher improves in high leverage situations",
             fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/overall_vs_twostrike_deviation.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Print the pitchers who improve most in two strike situations
comparison_data['two_strike_improvement'] = (
    comparison_data['two_strike_dev_cost'] - 
    comparison_data['deviation_advantage']
)
print("=== Pitchers Who Improve Most in Two Strike Situations ===")
print("(Positive = better two strike deviator than overall)")
print(comparison_data[['pitcher_name', 'deviation_advantage',
                        'two_strike_dev_cost', 'two_strike_improvement']]
      .sort_values('two_strike_improvement', ascending=False)
      .round(2).to_string(index=False))

## Hitter Count Vulnerability Ranking

In [ ]:
# Extract hitter count deviation costs for all pitchers
# This is a key batter exploitation metric feeding into Phase 4

hitter_count_dev = count_cat_analysis[
    count_cat_analysis['count_category'] == 'hitter_count'
].pivot_table(
    index='pitcher_name',
    columns='followed_recommendation',
    values='positive_rate'
).reset_index()

hitter_count_dev.columns = ['pitcher_name', 
                             'hitter_count_dev_rate',
                             'hitter_count_follow_rate']
hitter_count_dev['hitter_count_vulnerability'] = (
    hitter_count_dev['hitter_count_dev_rate'] - 
    hitter_count_dev['hitter_count_follow_rate']
).round(2)

hitter_count_dev = hitter_count_dev.sort_values('hitter_count_vulnerability')

print("=== Hitter Count Vulnerability Ranking ===")
print("Most negative = pitcher most vulnerable when deviating in hitter counts")
print(hitter_count_dev.round(2).to_string(index=False))

# Save for Phase 4 use
hitter_count_dev.to_parquet(
    '../data/processed/hitter_count_vulnerability.parquet',
    index=False
)
print("\nSaved to data/processed/hitter_count_vulnerability.parquet")